# Encapsulation in Python

**OOP-in-Python — Basics**  
Access modifiers, name mangling, properties, and getter/setter patterns.

## What is Encapsulation?

Encapsulation **bundles data with the methods that operate on it** and **restricts direct access** to an object's internal state.

Python uses **convention** rather than strict enforcement:

| Access Level | Convention | Example |
|---|---|---|
| Public | No underscore | `self.name` |
| Protected | Single underscore `_` | `self._internal` |
| Private | Double underscore `__` | `self.__secret` |

## 1. Public Members

Accessible from **anywhere** — no underscore prefix.

In [ ]:
class Person:
    def __init__(self, name: str):
        self.name = name  # Public — accessible everywhere

p = Person("Alice")
print(p.name)  # Alice — direct access is fine
p.name = "Bob" # Can be modified directly

## 2. Protected Members (Single Underscore `_`)

- **Convention**: "internal use" — not meant for external access.
- Still accessible — it's a **gentleman's agreement**.
- Subclasses may access it.

In [ ]:
class Employee:
    def __init__(self, name: str, salary: float):
        self.name = name
        self._salary = salary  # Protected — 'internal use' convention

    def _calculate_bonus(self) -> float:
        # Internal helper — not part of public API
        return self._salary * 0.1

class Manager(Employee):
    def bonus(self) -> float:
        # Subclass CAN access protected members
        return self._calculate_bonus() * 2

e = Employee("Alice", 60000)
print(e._salary)  # Works (but frowned upon)
m = Manager("Bob", 80000)
print(m.bonus())   # 16000.0

## 3. Private Members (Double Underscore `__`)

- Triggers **name mangling**: `__secret` becomes `_ClassName__secret`.
- Not truly private — but **harder to accidentally access**.
- Prevents subclasses from accidentally overriding.

In [ ]:
class BankAccount:
    def __init__(self, owner: str, balance: float):
        self.owner = owner
        self.__balance = balance  # Private — name mangled

    def get_balance(self) -> float:
        return self.__balance

    def deposit(self, amount: float):
        if amount > 0:
            self.__balance += amount

acc = BankAccount("Alice", 1000)
# print(acc.__balance)  # AttributeError!
print(acc.get_balance())  # 1000 — access through method

# Name mangling — the 'secret' way to access (but don't do this)
print(acc._BankAccount__balance)  # 1000 — accessible but strongly discouraged

### Name Mangling Details

Python **rewrites** `__attr` to `_ClassName__attr` at compile time.

In [ ]:
class A:
    def __init__(self):
        self.__secret = 42

class B(A):
    def __init__(self):
        super().__init__()
        self.__secret = 99  # Different attribute! _B__secret

b = B()
print(b._A__secret)  # 42 — parent's version
print(b._B__secret)  # 99 — child's version
print(dir(b))        # Show all attributes with mangled names

## 4. Properties (`@property`)

The **Pythonic** way to implement getters and setters. Allows you to:
- Start with a simple attribute
- Later add validation without breaking the API
- Use `@property` (getter), `@x.setter` (setter), `@x.deleter` (deleter)

In [ ]:
class Temperature:
    def __init__(self, celsius: float = 0.0):
        self._celsius = celsius  # Internal storage

    @property
    def celsius(self) -> float:
        # Getter
        return self._celsius

    @celsius.setter
    def celsius(self, value: float):
        # Setter with validation
        if value < -273.15:
            raise ValueError("Temperature below absolute zero!")
        self._celsius = value

    @celsius.deleter
    def celsius(self):
        print("Deleting temperature")
        del self._celsius

    @property
    def fahrenheit(self) -> float:
        # Computed property — no setter (read-only)
        return (self._celsius * 9 / 5) + 32

t = Temperature(100)
print(t.celsius)     # 100 — looks like attribute access
print(t.fahrenheit)  # 212.0 — computed on the fly
t.celsius = 0
print(t.fahrenheit)  # 32.0
# t.celsius = -300   # ValueError!
del t.celsius        # Deleting temperature

## 5. Property vs Direct Attribute — Pattern Summary

**Without properties** (Java-style getters/setters — unpythonic):
```python
def get_name(self): return self._name
def set_name(self, value): self._name = value
```

**With properties** (Pythonic):
```python
@property
def name(self): return self._name

@name.setter
def name(self, value): self._name = value
```

The caller sees `obj.name` — no parentheses, no API change if we add logic later.

---
## Problem 1: Temperature Class with Conversion

**Difficulty: Easy**

Build a `Temperature` class that:
- Stores temperature internally in Celsius
- Provides `celsius` and `fahrenheit` properties
- Validates that values stay above absolute zero
- Allows setting via either unit

In [ ]:
class Temperature:
    def __init__(self, initial_celsius: float = 0.0):
        self._celsius = initial_celsius

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float):
        if value < -273.15:
            raise ValueError("Below absolute zero!")
        self._celsius = value

    @property
    def fahrenheit(self) -> float:
        return round(self._celsius * 9 / 5 + 32, 2)

    @fahrenheit.setter
    def fahrenheit(self, value: float):
        # Convert Fahrenheit to Celsius for storage
        celsius_val = (value - 32) * 5 / 9
        if celsius_val < -273.15:
            raise ValueError("Below absolute zero!")
        self._celsius = celsius_val

    def __repr__(self) -> str:
        return f"Temperature({self._celsius:.2f}°C / {self.fahrenheit:.2f}°F)"

t = Temperature()
t.celsius = 25
print(t)                 # Temperature(25.00°C / 77.00°F)
t.fahrenheit = 100
print(t)                 # Temperature(37.78°C / 100.00°F)

try:
    t.celsius = -300
except ValueError as e:
    print(f"Error: {e}")  # Error: Below absolute zero!

## Problem 2: BankAccount with Balance Validation

**Difficulty: Medium**

Create a `BankAccount` with:
- Properties for balance with validation (no negative)
- Read-only account number (generated)
- Deposit/withdraw methods
- Transaction log (protected)

In [ ]:
class BankAccount:
    _next_acc_no = 10000

    def __init__(self, owner: str, initial_balance: float = 0.0):
        self.owner = owner
        self._account_no = BankAccount._next_acc_no
        BankAccount._next_acc_no += 1
        self._balance = 0.0  # Will set via setter
        self.balance = initial_balance  # Trigger setter validation
        self._transactions = []  # Protected transaction log

    @property
    def account_no(self) -> int:
        return self._account_no  # Read-only — no setter

    @property
    def balance(self) -> float:
        return self._balance

    @balance.setter
    def balance(self, value: float):
        if value < 0:
            raise ValueError("Balance cannot be negative")
        self._balance = value

    def deposit(self, amount: float) -> float:
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount
        self._transactions.append(f"+{amount}")
        return self._balance

    def withdraw(self, amount: float) -> float:
        if amount <= 0:
            raise ValueError("Withdrawal must be positive")
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
        self._transactions.append(f"-{amount}")
        return self._balance

    @property
    def transaction_log(self) -> list:
        return list(self._transactions)  # Return a copy to protect internal state

    def __str__(self) -> str:
        return f"Account #{self._account_no} — {self.owner}: ${self._balance:.2f}"

# Test
acc = BankAccount("Alice", 1000)
print(acc)
acc.deposit(500)
acc.withdraw(200)
print(f"Balance: ${acc.balance}")
print(f"Transactions: {acc.transaction_log}")
# acc.balance = -50  # ValueError!
# acc.account_no = 999  # AttributeError (no setter)

## Key Takeaways

- **Public**: no underscore — accessible everywhere.
- **Protected** (`_`): convention-based, for internal/subclass use.
- **Private** (`__`): name mangling to `_ClassName__attr` — avoids accidental access/subclass clashes.
- **Properties** (`@property`): Pythonic getters/setters — add validation without breaking the API.
- Encapsulation is about **hiding implementation details** and exposing a clean interface.